# 04 · Three-stage cascade on the sealed test split

Runs Stage 1 → 2 → 3 once on the test split with hard gating: an image rejected by
a gate receives zero probability for every downstream target. Reports the
end-to-end metrics and the **loss cascade**, which attributes every missed disease
instance to the stage that lost it.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

In [ ]:
from fundus.models import build_classifier, load_weights
from fundus.inference import run_cascade, loss_cascade
from fundus.metrics import score_all, per_class_table

te = load_split(P.csv("test"))
R = P.run_root
S1_THRESHOLD = json.load(open(R / "stage1/operating_point.json"))["threshold"]
S2_THR = json.load(open(R / "stage2_bce/operating_points.json"))["thresholds"]
S3_DIRS = {"G1": "G1", "G2": "G2", "G3": "G3", "G4": "G4", "G5": "G5_neg", "G6": "G6"}

import dataclasses
cfg = dataclasses.replace(config.STAGE3, pretrained=False)   # weights come from checkpoints
s1 = load_weights(build_classifier(2, cfg), R / "stage1/best_val_auc.pt", DEV)
s2 = load_weights(build_classifier(len(labels.GROUP_COLS), cfg), R / "stage2_bce/best_macro_auc.pt", DEV)
s3, s3_thr = {}, {}
for k, d in S3_DIRS.items():
    n = len(labels.GROUPS[labels.GKEY[k]])
    s3[k] = load_weights(build_classifier(n, cfg), R / "stage3" / d / "best_macro_ap.pt", DEV)
    s3_thr[k] = json.load(open(R / "stage3" / d / "thresholds.json"))["thresholds"]

In [ ]:
final_prob, final_pred, s1_tab, s2_tab = run_cascade(
    te, P.images("test"), s1, s2, s3, S1_THRESHOLD, S2_THR, s3_thr, DEV)

Y = te.set_index("image").loc[final_pred.index, labels.TARGETS].values
res = score_all(Y, final_prob.values, 0.5, "3-stage pipeline (probabilities @0.5)")
# the cascade's own decisions (per-class Stage-3 thresholds):
res_native = score_all(Y, final_pred.values.astype(float), 0.5, "3-stage pipeline (native decisions)")
print(pd.DataFrame([res, res_native]).set_index("config").round(4).T)

cascade = loss_cascade(te, final_pred, s1_tab, s2_tab)
T = cascade["total"]
for k in ["lost_stage1", "lost_stage2", "lost_stage3", "found"]:
    print(f"{k:12s} {cascade[k]:5d}  ({cascade[k] / T:.1%})")

In [ ]:
OUT = P.run("pipeline_test")
final_prob.to_csv(OUT / "pipeline_test_probabilities.csv", index_label="image")
final_pred.to_csv(OUT / "pipeline_test_predictions.csv", index_label="image")
per_class_table(Y, final_pred.values.astype(float), 0.5, labels.TARGETS).to_csv(OUT / "pipeline_per_disease.csv")
json.dump({"metrics": res, "metrics_native": res_native, "loss_cascade": cascade},
          open(OUT / "pipeline_summary.json", "w"), indent=2)